# Notebook 0 – F2 Platform and Lab Environment Setup

> ⚠️ **Cost reminder:** When working with this notebook in a workshop environment, remember to stop or terminate the F2 instance from the [EC2 Console](https://console.aws.amazon.com/ec2/) at the end of your session to avoid unnecessary costs.

This notebook introduces the F2 platform, the software stack, and the Jupyter environment used throughout this notebook series. By the end, the FPGA will be verified as visible and the SDK confirmed as working.

## Table of Contents

**Part I – F2 Background**
- [1. F2 Architecture Recap](#1.-F2-Architecture-Recap)
- [2. Software Stack](#2.-Software-Stack)
- [3. Key Vocabulary](#3.-Key-Vocabulary)

**Part II – Jupyter Essentials**
- [4. Running Cells](#4.-Running-Cells)
- [5. Execution Order and Kernel State](#5.-Execution-Order-and-Kernel-State)
- [6. Importing from .py Files](#6.-Importing-from-.py-Files)
- [7. Shell Commands and sudo](#7.-Shell-Commands-and-sudo)
- [8. Environment Verification](#8.-Environment-Verification)

---
# Part I – F2 Background

## 1. F2 Architecture Recap

This section recaps the key F2 concepts covered in this notebook. For detailed documentation, see the [official F2 documentation](https://awsdocs-fpga-f2.readthedocs-hosted.com).

### 1.1. F2 at a Glance

**Amazon EC2 F2 instances** pair a host CPU (**3rd Gen AMD EPYC**) with one or more **Xilinx Virtex UltraScale+ VU47P** FPGAs, connected over a **PCIe Gen4 x8** link. The CPU and FPGA are physically separate devices with separate memory spaces and clock domains; all communication between them crosses the PCIe bus.

**Host and FPGA domains.** F2 development spans two domains. The **host** (the CPU and its associated system memory) runs a Linux-based OS and executes conventional software (Python, C, drivers). The **FPGA** runs custom digital logic written in HDL (SystemVerilog / VHDL), known as **RTL** (Register Transfer Level) design, where circuits operate in parallel, synchronized to a clock.

**Communication between domains.** These two domains communicate over **PCIe**. AWS provides abstraction layers so that you do not need to construct raw PCIe packets on either side. From the software side, you call high-level functions. From the hardware side, you implement standard bus interfaces. The F2 Shell provides the well-defined contract between them.

**Shell and Custom Logic.** Each FPGA on an F2 instance is logically divided into two regions:

- **Shell (SH)**: The AWS-provided platform logic (PCIe endpoint, interrupts, memory controllers, clocks). The Shell is fixed and versioned by AWS.

- **Custom Logic (CL)**: The accelerator design, connected to the Shell through standard interfaces. In addition to RTL, F2 supports higher-level development flows including Vitis and HLx. For more details, see the [official documentation](https://awsdocs-fpga-f2.readthedocs-hosted.com).

Together, Shell + CL are compiled into an **Amazon FPGA Image (AFI)**, a signed, encrypted bitstream that can be loaded onto the FPGA at runtime.

### 1.2. How Host Software Talks to the FPGA: The Interfaces

The Shell exposes several interfaces between the host and the Custom Logic. All are defined in the HDK file `cl_ports.vh` and are synchronous to the primary clock `clk_main_a0` at 250 MHz.

| Interface | Protocol | Data Width | PCIe Mapping | Primary Purpose |
|-----------|----------|------------|--------------|------------------|
| **OCL** | AXI-Lite | 32-bit | AppPF BAR0 (64 MiB) | Register reads and writes from the host (memory-mapped I/O for control and status) |
| **SDA** | AXI-Lite | 32-bit | MgmtPF BAR4 (4 MiB) | Management and configuration registers, clock generator access |
| **PCIS** | AXI-4 | 512-bit | AppPF BAR4 (128 GiB) | Bulk inbound data transfers: host writes large blocks to FPGA-attached memory (DDR/HBM) |
| **PCIM** | AXI-4 | 512-bit | CL-initiated | Bulk outbound data transfers: FPGA reads from or writes to host memory |

Beyond these data interfaces, the Shell also provides Clocks, Resets, Interrupts (16 MSI-X sources), Virtual LEDs, Virtual DIP switches, HBM Monitor APB, and DDR Stats connections to the CL. For the complete interface specification, see the [AWS Shell Interface Specification](https://awsdocs-fpga-f2.readthedocs-hosted.com/latest/hdk/docs/AWS-Shell-Interface-Specification.html).

Each interface maps to a **PCIe BAR** (Base Address Register), a region of the FPGA's PCIe address space accessible as memory from the host. The Shell translates PCIe transactions into AXI bus transactions that the CL receives. For details on the full PCIe memory map, see the [FPGA PCIe Memory Map](https://awsdocs-fpga-f2.readthedocs-hosted.com/latest/hdk/docs/AWS-Fpga-Pcie-Memory-Map.html).

**Key insight for software developers**: When you call `pci_poke(handle, address, value)` in Python, the SDK, kernel driver, PCIe hardware, and Shell collaborate to deliver that value to the CL as an AXI-Lite write transaction. The entire PCIe protocol stack is invisible to your application.

**Key insight for hardware developers**: You implement an AXI-Lite slave module in the CL. PCIe Transaction Layer Packets (TLPs), BAR configuration, and bus enumeration are all handled by the Shell.

## 2. Software Stack

This section describes the software environment used across all notebooks.

### 2.1. Execution Environment

This notebook runs **directly on the F2 instance**, on the host CPU. The Python kernel executes on the same Linux operating system that has PCIe access to the FPGA. Each code cell executes code that is **directly communicating with the FPGA hardware**. The register values read back come from actual flip-flops in the FPGA fabric, toggling at 250 MHz.

### 2.2. The Layers

Between the Python code and the FPGA hardware, there are several layers of abstraction. Each layer translates an operation into a form the next layer understands:

| Layer | Details |
|-------|--------|
| Runtime Software (this Jupyter notebook) | Python code, or a C/C++ application |
| Cython Bindings (Python to C bridge) | fpga_mgmt_wrapper, fpga_pci_wrapper, fpga_clkgen_wrapper |
| C SDK Libraries (fpga_pci_poke, fpga_pci_peek, etc.) | libfpga_mgmt.so, libfpga_pci.so |
| Linux Kernel + FPGA Drivers | FPGA management + application PF drivers |
| PCIe Hardware Interface | Gen4 x8 physical link |
| FPGA Shell + Custom Logic | The CL (RTL) resides here |

All intermediate layers, from the C SDK through the Shell, are provided by AWS, reducing the complexity of hardware/software co-design.

> Note: The Cython bindings layer is specific to Python development. C/C++ applications call the SDK libraries directly, skipping this layer.

Reference: [F2 SDK Documentation](https://awsdocs-fpga-f2.readthedocs-hosted.com/latest/sdk/README.html)

### 2.3. Cython Bindings

The AWS FPGA SDK is written in C. The **Cython bindings** wrap these C functions into Python classes: **FpgaMgmt** (AFI management), **FpgaPCI** (register read/write), and **FpgaClkgen** (clock control), giving Python direct access with negligible overhead. All FPGA operations require **root privileges**; the Jupyter server for this notebook is configured accordingly.

Reference: [Python Bindings Documentation](https://awsdocs-fpga-f2.readthedocs-hosted.com/latest/sdk/userspace/cython-bindings/README.html)

## 3. Key Vocabulary

The following terms are used throughout this notebook and all subsequent notebooks:

| Term | Definition |
|------|------------|
| **Shell (SH)** | AWS-provided platform logic on the FPGA (PCIe endpoint, memory controllers, clocks). Fixed and versioned by AWS. |
| **CL (Custom Logic)** | The accelerator design on the FPGA, connected to the Shell through defined interfaces. |
| **OCL** | The AXI-Lite interface mapped to AppPF BAR0, used for CL register reads/writes from the host. |
| **Slot** | Each FPGA in the instance is identified by a zero-indexed slot number. |
| **PF (Physical Function)** | A distinct addressable device exposed by a single physical PCIe card. Each FPGA slot exposes two PFs: **AppPF** (PF0, application) and **MgmtPF** (AWS management). |
| **BAR (Base Address Register)** | A PCIe memory region mapped into the host's virtual address space. The OCL interface maps to AppPF BAR0. The PCIS interface maps to AppPF BAR4. |
| **Handle** | An integer token returned by `pci_attach()` representing a connection to a specific PF and BAR combination. All subsequent poke and peek calls require this handle. Call `pci_detach()` when finished. |
| **Poke** | A 32-bit write to a register at a given byte offset within the attached BAR. On the FPGA side, this becomes an AXI-Lite write transaction. |
| **Peek** | A 32-bit read from a register at a given byte offset within the attached BAR. On the FPGA side, this becomes an AXI-Lite read transaction. |
| **AFI / AGFI** | Amazon FPGA Image (regional identifier) / Amazon Global FPGA Image ID (global identifier used for loading onto FPGA slots). |


---
# Part II – Jupyter Essentials

Readers already familiar with Jupyter notebooks can skip to [Section 8 (Environment Verification)](#8.-Environment-Verification). Otherwise, work through each section; the patterns covered here are used throughout this notebook series.

## 4. Running Cells

A Jupyter notebook is a sequence of **cells**. Each cell is either **Markdown** (formatted text, like this one) or **Code** (executable Python). Code cells behave like small, self-contained scripts: write a few lines, execute them immediately, and inspect the output inline. This makes notebooks well suited for interactive hardware exploration — read a register, inspect the result, adjust, and repeat, without recompiling or restarting anything.

To execute a code cell:

- Select it and press **Shift + Enter** (executes the cell and advances to the next), or
- Click the **Run** button in the toolbar.

After execution, a number appears in the left margin (e.g., `In [1]`). This is the **execution counter** — it reflects the global order in which cells were run, not their position in the notebook. This distinction becomes important in the next section.

Execute the cell below and observe the counter:

In [ ]:
print("Hello from F2! 🚀")

## 5. Execution Order and Kernel State

All code cells share a single Python **kernel** (process). Variables, imports, and function definitions created in one cell persist in memory and are visible to every other cell. This is powerful but carries two important consequences:

1. **State accumulates.** Executing a cell changes the kernel state, and executing it again changes it further.
2. **Order is determined by when cells are run, not where they sit.** Executing cells out of top-to-bottom order can produce unexpected results.

### 5.1. State Accumulates

Execute the increment cell below multiple times and observe the counter grow. Each execution reads `counter`, adds 1, and stores the result back. The kernel retains the value between runs.

In [ ]:
counter = 0
print(f"{counter = }")

In [ ]:
# Run this cell multiple times — the counter keeps incrementing
counter = counter + 1
print(f"{counter = }")

The kernel retained the value of `counter` between executions. Running the increment cell without first running the initialization cell — for example after a kernel restart — will raise a `NameError` since `counter` no longer exists.

### 5.2. Out-of-Order Execution

Execute the next two cells in order (Cell A first, then Cell B):

In [ ]:
# Cell A: define the message
message = "FPGA says hello"

In [ ]:
# Cell B: use the message
print(message.upper())

To observe out-of-order failure:

1. **Restart the kernel** (**Kernel → Restart**).
2. Execute **only Cell B** (skip Cell A).

The kernel raises `NameError: name 'message' is not defined`, since `message` was defined in Cell A, which was not executed. The kernel holds no state from previous sessions.

This is the most common source of errors in notebook workflows. In these notebooks, setup cells (imports, constants, FPGA initialization) are always placed at the top. **Kernel → Restart & Run All** is the recommended recovery.

## 6. Importing from .py Files

Not all code belongs in notebook cells. Helper functions can live in `.py` files alongside the notebooks, imported as standard Python modules.

The file `notebook_utils.py` sits alongside this notebook, with utility functions used across the notebooks. Import a function from it and execute it:

In [ ]:
from helpers.notebook_utils import fmt_hex32

print(fmt_hex32(255))           # 0x000000FF
print(fmt_hex32(0xDEADBEEF))    # 0xDEADBEEF

## 7. Shell Commands and sudo

Prefix a line with `!` to run a **shell command** directly from a notebook cell. This is useful for system-level checks, as verification cells can be added without leaving the notebook.

All FPGA management tools (`fpga-describe-local-image-slots`, `fpga-load-local-image`, etc.) require **root privileges**. The Jupyter server for this notebook runs as root, so `sudo` is not needed inside notebook cells. On a standard setup, FPGA commands must be prefixed with `sudo`.

Execute the cell below to query the FPGA slot status:

In [ ]:
!fpga-describe-local-image -S 0 -H

The output shows the FPGA slot status, the loaded AFI (if any), and the Shell version. The `-H` flag formats the output as a human-readable table. This command is used throughout the notebooks to verify AFI loads and check for errors.

## 8. Environment Verification

The cells below confirm that the Cython bindings are installed and the FPGA slot is accessible. A successful run indicates the environment is ready for all subsequent notebooks.

In [ ]:
from fpga_mgmt_wrapper import FpgaMgmt
from fpga_pci_wrapper import FpgaPCI

print("✅ Cython bindings imported successfully")

> **If the cell above raises `ModuleNotFoundError: No module named 'fpga_mgmt_wrapper'`**: run `source sdk_setup.sh` from a terminal at the repo root (if using this notebook, use the VSCode server terminal — it opens at the repo root by default), then restart the kernel (**Kernel → Restart**) and re-run from the top.

In [ ]:
import json

from helpers.notebook_utils import print_fpga_info

mgmt = FpgaMgmt()
info = mgmt.describe_local_image(slot_id=0)
print_fpga_info(info)
print(f"\n✅ FPGA slot 0 is accessible — status: {info['status']}")

The `status` field will show `cleared` (no AFI loaded) or `loaded` (an AFI is present). Either is acceptable. The important result is that the call succeeded, confirming the SDK, driver, and FPGA hardware are all operational.

If the cell above raised an error, verify that:
- The notebook is running on an F2 instance (not a standard EC2 instance).
- The SDK was built (`source sdk_setup.sh` from the repo root).
- The Jupyter server has root privileges.

---

The environment is verified and ready. Proceed to **Notebook 1 – CL_AXIL_REG_ACCESS** to load the first AFI and interact with FPGA registers.

> ⚠️ **Cost reminder:** When working with this notebook in a workshop environment, remember to stop or terminate the F2 instance from the [EC2 Console](https://console.aws.amazon.com/ec2/) at the end of your session to avoid unnecessary costs.